# Reciter TTS — Silero v5.5 (RU) → PyTorch Mobile (.ptl) → Android

Silero v5 принимает **сырой текст** и сам делает ударения/омографы/числа (фронтенд зашит в JIT-модель). ONNX-экспорта нет, поэтому на устройстве используем **PyTorch Mobile**: модель `.ptl` выполняет `apply_tts` целиком.

Маленькая, быстрее реального времени на CPU, нативная кириллица, готовые дикторы (aidar, baya, kseniya, xenia, eugene).

На выходе — ZIP для импорта в приложении.


## Ячейка 1: Зависимости


In [ ]:
!pip install -q torch torchaudio soundfile
import torch, os, json, zipfile
print("torch", torch.__version__)

## Ячейка 2: Загрузка Silero v5_5_ru


In [ ]:
SPEAKER_PKG = "v5_ru"     # пакет ru-моделей v5 (включает v5_5)
device = torch.device("cpu")
model, example_text = torch.hub.load(repo_or_dir="snakers4/silero-models",
                                     model="silero_tts", language="ru", speaker=SPEAKER_PKG)
model.to(device)
print("пример текста:", example_text[:80])
print("методы модели:", [m for m in dir(model) if not m.startswith("_")][:40])
# дикторы
SPEAKERS = ["aidar","baya","kseniya","xenia","eugene"]
SR = 48000

## Ячейка 3: Проверка apply_tts (должна звучать русская речь с ударениями)


In [ ]:
import soundfile as sf
from IPython.display import Audio, display
txt = "Привет! Это проверка синтеза речи на русском языке."
audio = model.apply_tts(text=txt, speaker="xenia", sample_rate=SR)
sf.write("/content/silero_test.wav", audio.numpy(), SR)
print("длительность:", len(audio)/SR, "с")
display(Audio(audio.numpy(), rate=SR))

## Ячейка 4: Сохранить для PyTorch Mobile (lite) и проверить загрузку
Если lite-модель грузится и `apply_tts` работает — на Android заведётся.


In [ ]:
from torch.utils.mobile_optimizer import optimize_for_mobile
OUT = "/content/silero-android"; os.makedirs(OUT, exist_ok=True)

# Silero отдаёт уже scripted-модель (torch.jit). Сохраняем для lite-интерпретатора.
ptl = f"{OUT}/silero_v5_5_ru.ptl"
try:
    model._save_for_lite_interpreter(ptl)
    print("сохранено через _save_for_lite_interpreter")
except Exception as e:
    print("прямое сохранение не вышло:", e, "-> пробуем jit.script")
    sm = torch.jit.script(model)
    sm._save_for_lite_interpreter(ptl)
print("ptl:", os.path.getsize(ptl)/1024**2, "MB")

# проверка: грузим .ptl обратно обычным jit.load и зовём apply_tts
chk = torch.jit.load(ptl)
a2 = chk.apply_tts(text="Проверка лайт модели.", speaker="baya", sample_rate=SR)
print("lite apply_tts OK, сэмплов:", a2.shape)

## Ячейка 5: Манифест + архив


In [ ]:
manifest = {
  "schemaVersion": 1, "id": "silero-v5-ru", "displayName": "Silero v5.5 (Russian)",
  "family": "silero", "architecture": "silero", "sampleRateHz": SR,
  "runtime": "pytorch-lite",
  "tokenizer": {"type": "builtin", "files": []},
  "files": [{"filename": "silero_v5_5_ru.ptl", "role": "TALKER",
             "sizeMb": round(os.path.getsize(ptl)/1024**2), "required": True}],
  "voices": [{"id": s, "locale": "ru-RU", "displayName": s.capitalize(), "speakerId": i}
             for i, s in enumerate(SPEAKERS)],
}
json.dump(manifest, open(f"{OUT}/model.json","w"), ensure_ascii=False, indent=2)

zip_path = "/content/silero-v5-ru-android.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, f), f"models/{f}")
print("архив:", round(os.path.getsize(zip_path)/1024**2), "MB")
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass